# 02 — Business-oriented exploratory analysis

**North star question:** *What drives movie commercial success before release?*

This notebook treats `movies_cleaned_with_target.csv` as the business dataset: we explore
patterns in **pre-release signals** vs **commercial outcomes** (ROI and success class), form
**investor-style narratives**, save **publication-ready figures** under `plots/business_eda/`
for reuse in Streamlit later, and record **modeling hypotheses** — without training models.

**How to read each section:** charts and tables first, then the short **Insights** block
(observation → interpretation → modeling implication).


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Seaborn 0.13 + matplotlib récents : `sns.boxplot` appelle encore `ax.bxp(vert=...)`, ce qui déclenche un
# avertissement (souvent PendingDeprecationWarning, parfois FutureWarning selon les versions).
# Filtre strictement limité à ce message — pas un filtre global.
for _cat in (PendingDeprecationWarning, FutureWarning):
    warnings.filterwarnings(
        "ignore",
        message=r"vert: bool will be deprecated.*",
        category=_cat,
    )

_CWD = Path.cwd().resolve()
if (_CWD / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD
elif (_CWD.parent / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD
    print("Warning: data/processed not found next to cwd or parent; using cwd.")

CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "movies_cleaned_with_target.csv"
PLOTS_DIR = PROJECT_ROOT / "plots" / "business_eda"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk", font_scale=0.95)
plt.rcParams["figure.figsize"] = (11, 5.5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

def save_fig(name: str) -> Path:
    path = PLOTS_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Dataset exists:", CLEAN_PATH.exists())
print("Figures will save to:", PLOTS_DIR)



In [ ]:
df = None
if not CLEAN_PATH.exists():
    print(f"Missing: {CLEAN_PATH}")
    print("Run notebooks/01_dataset_audit_and_target.ipynb first to create the cleaned CSV.")
else:
    df = pd.read_csv(CLEAN_PATH)
    if "roi" not in df.columns:
        raise ValueError("Expected column 'roi' in cleaned dataset.")
    df["log_roi"] = np.log1p(df["roi"])
    if "movie_success_class" in df.columns:
        df["movie_success_class"] = pd.Categorical(
            df["movie_success_class"],
            categories=["flop", "average", "hit"],
            ordered=True,
        )
    print("shape:", df.shape)
    display(df.head(3))
    display(df[["budget", "revenue", "roi", "log_roi", "movie_success_class"]].describe(include="all"))



## 1. ROI distribution (raw vs log)

**Business lens:** ROI is highly skewed in entertainment finance; log scale highlights the
**typical** return while raw scale exposes **blockbusters** and **disasters**.


In [ ]:
if df is None:
    print("Skip: no dataframe.")
else:
    roi = df["roi"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(roi, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
    axes[0].set_title("ROI distribution (raw)")
    axes[0].set_xlabel("ROI (revenue / budget)")
    axes[0].set_ylabel("Count")

    axes[1].hist(df["log_roi"], bins=60, color="darkslateblue", edgecolor="white", alpha=0.85)
    axes[1].set_title("log(1 + ROI) distribution")
    axes[1].set_xlabel("log_roi")
    axes[1].set_ylabel("Count")
    plt.tight_layout()
    save_fig("01_roi_raw_and_log")

    q01, q99 = roi.quantile(0.01), roi.quantile(0.99)
    iqr = roi.quantile(0.75) - roi.quantile(0.25)
    low_iqr = roi.quantile(0.25) - 1.5 * iqr
    high_iqr = roi.quantile(0.75) + 1.5 * iqr
    extreme_high = int((roi > max(high_iqr, q99)).sum())
    extreme_low = int((roi < max(low_iqr, 0)).sum())
    print("ROI percentiles p01 / p50 / p99:", float(q01), float(roi.median()), float(q99))
    print("Approx. extreme high tail count (above ~IQR or p99):", extreme_high)
    print("Approx. extreme low tail count:", extreme_low)



### Insights — ROI distribution

- **Observation:** Raw ROI is long-right-tailed; `log_roi` compresses extremes for comparison.
- **Business interpretation:** A small set of outsized hits and some deep losses drive the tail;
  medians and interquartile ranges often matter more than means for greenlight conversations.
- **Modeling implication:** Prefer robust models / stratified evaluation; consider log-transform
  or quantile features for **regression** on ROI; keep **classification** on `movie_success_class`
  aligned with stakeholder language.


## 2. Success class distribution

**Business lens:** Portfolio mix (flop / average / hit) sets expectations for precision/recall
trade-offs in a decision-support tool.


In [ ]:
if df is None:
    print("Skip.")
else:
    vc = df["movie_success_class"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8, 5))
    x = vc.index.astype(str)
    ax.bar(x, vc.values, color="#2c699a", edgecolor="white", linewidth=0.8)
    ymax = float(vc.values.max())
    ax.set_ylim(0, ymax * 1.12)
    ax.set_title("Commercial success classes (ROI-based)")
    ax.set_xlabel("movie_success_class")
    ax.set_ylabel("Count")
    for i, v in enumerate(vc.values):
        ax.text(i, v + max(vc.values) * 0.01, str(int(v)), ha="center", fontsize=11)
    save_fig("02_success_class_counts")

    props = (vc / vc.sum()).rename("share")
    display(pd.concat([vc.rename("count"), props], axis=1))




### Insights — success classes

- **Observation:** Class balance shows whether “hit” is a rare event (often yes).
- **Business interpretation:** Rare hits can dominate upside; the product story may emphasize
  **ranking** or **probability of average-or-better**, not only hit detection.
- **Modeling implication:** Plan for imbalance (class weights, stratified CV, macro-F1 reporting).


## 3. Budget vs outcomes

**Business lens:** “Does spending more buy a better ROI?” — the core tension between
**scale** and **efficiency**.


In [ ]:
if df is None:
    print("Skip.")
else:
    sub = df[(df["budget"] > 0) & (df["revenue"] > 0)].copy()

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(np.log1p(sub["budget"]), np.log1p(sub["revenue"]), alpha=0.25, s=18, c="steelblue")
    ax.set_title("Budget vs revenue (log1p scale)")
    ax.set_xlabel("log1p(budget)")
    ax.set_ylabel("log1p(revenue)")
    save_fig("03a_budget_vs_revenue_log")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(np.log1p(sub["budget"]), sub["log_roi"], alpha=0.25, s=18, c="darkslateblue")
    ax.set_title("Budget vs log ROI")
    ax.set_xlabel("log1p(budget)")
    ax.set_ylabel("log_roi")
    save_fig("03b_budget_vs_log_roi")

    sub["budget_quartile"] = pd.qcut(sub["budget"], q=4, labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"], duplicates="drop")
    fig, ax = plt.subplots(figsize=(10, 5.5))
    sns.boxplot(data=sub, x="budget_quartile", y="roi", hue="budget_quartile", palette="Blues", dodge=False, legend=False, ax=ax)
    ax.set_title("ROI by budget quartile (raw ROI — expect outliers)")
    ax.set_xlabel("Budget quartile")
    ax.set_ylabel("ROI")
    save_fig("03c_roi_by_budget_quartile")

    fig, ax = plt.subplots(figsize=(10, 5.5))
    sns.violinplot(data=sub, x="budget_quartile", y="log_roi", hue="budget_quartile", palette="muted", dodge=False, inner="box", ax=ax, legend=False)
    ax.set_title("log ROI by budget quartile")
    ax.set_xlabel("Budget quartile")
    ax.set_ylabel("log_roi")
    save_fig("03d_logroi_by_budget_quartile_violin")

    fig, ax = plt.subplots(figsize=(10, 5.5))
    sub["log_budget"] = np.log1p(sub["budget"])
    sns.boxplot(data=sub, x="movie_success_class", y="log_budget", hue="movie_success_class", palette="Blues", dodge=False, legend=False, ax=ax)
    ax.set_title("log1p(budget) distribution by success class")
    ax.set_xlabel("movie_success_class")
    ax.set_ylabel("log1p(budget)")
    save_fig("03e_logbudget_by_success_class")

    med = sub.groupby("budget_quartile", observed=True)["roi"].median()
    print("Median ROI by budget quartile:")
    display(med.to_frame("median_roi"))




### Insights — budget

- **Observation:** Scatter and quartile medians show whether higher budgets associate with
  higher **revenue** and whether **ROI efficiency** holds at scale.
- **Business interpretation:** Big budgets can reduce relative ROI (denominator effect) while
  still lifting absolute revenue — “success” depends on which KPI investors optimize.
- **Modeling implication:** Keep `budget` as a strong baseline feature; consider interactions
  with genre or release window in a later feature pass (still pre-release).


## 4. Genre signals

**Business lens:** Genres proxy **audience positioning** and risk profile before marketing outcomes exist.


In [ ]:
if df is None or "main_genre" not in df.columns:
    print("Skip (needs main_genre).")
else:
    g = df.dropna(subset=["main_genre"]).copy()
    top_genres = g["main_genre"].value_counts().head(12).index
    g12 = g[g["main_genre"].isin(top_genres)]

    fig, ax = plt.subplots(figsize=(10, 5))
    order = g12["main_genre"].value_counts().index
    sns.countplot(data=g12, y="main_genre", order=order, color="steelblue", ax=ax)
    ax.set_title("Most common main genres (top 12)")
    ax.set_xlabel("Count")
    save_fig("04a_main_genre_counts")

    min_n = 25
    vc = g["main_genre"].value_counts()
    keep = vc[vc >= min_n].index
    gf = g[g["main_genre"].isin(keep)]
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.boxplot(data=gf, x="log_roi", y="main_genre", order=sorted(keep), hue="main_genre", palette="Set2", dodge=False, legend=False, ax=ax)
    ax.set_title(f"log ROI by main_genre (genres with n ≥ {min_n})")
    ax.set_xlabel("log_roi")
    save_fig("04b_logroi_by_main_genre")

    ct = pd.crosstab(gf["main_genre"], gf["movie_success_class"], normalize="index") * 100
    fig, ax = plt.subplots(figsize=(10, 6))
    ct.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
    ax.set_title("Success class mix by main genre (% within genre)")
    ax.set_xlabel("main_genre")
    ax.set_ylabel("Share (%)")
    ax.legend(title="class", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    save_fig("04c_success_mix_by_genre")

    hit = (g["movie_success_class"] == "hit").astype(float)
    hit_rate = (
        g.assign(_hit=hit)
        .groupby("main_genre", observed=True)["_hit"]
        .mean()
        .mul(100)
        .sort_values(ascending=False)
    )
    print("Hit-rate % by main_genre (all genres, noisy for small n):")
    display(hit_rate.head(15).to_frame("hit_rate_pct"))




### Insights — genre

- **Observation:** Counts show mainstream genres; boxplots show **dispersion** of returns within genre.
- **Business interpretation:** Some genres combine **frequency** with **volatile** ROI (high risk / high reward).
- **Modeling implication:** Encode `main_genre` categorically; consider multi-label features later from raw `genres` JSON.


## 5. Release timing (seasonality)

**Business lens:** Release windows compete for attention; calendar effects can proxy competitive intensity.


In [ ]:
if df is None:
    print("Skip.")
else:
    if "release_month" not in df.columns:
        print("Missing release_month — run notebook 01 first.")
    else:
        m = df.dropna(subset=["release_month"]).copy()
        m["release_month"] = m["release_month"].astype(int)

        fig, ax = plt.subplots(figsize=(10, 5))
        order_m = list(range(1, 13))
        sns.countplot(data=m, x="release_month", order=order_m, color="teal", ax=ax)
        ax.set_title("Release volume by calendar month")
        ax.set_xlabel("Month")
        ax.set_ylabel("Count")
        save_fig("05a_release_month_volume")

        med_m = m.groupby("release_month", observed=True)["log_roi"].median().reindex(order_m)
        plot_m = med_m.dropna()
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.barplot(x=plot_m.index.astype(int), y=plot_m.values, color="cadetblue", ax=ax)
        ax.set_title("Median log ROI by release month")
        ax.set_xlabel("Month")
        ax.set_ylabel("Median log_roi")
        save_fig("05b_median_logroi_by_month")

        if "release_quarter" in m.columns:
            fig, ax = plt.subplots(figsize=(8, 5))
            sns.boxplot(data=m, x="release_quarter", y="log_roi", hue="release_quarter", palette="crest", dodge=False, legend=False, ax=ax)
            ax.set_title("log ROI by fiscal-style quarter")
            ax.set_xlabel("release_quarter")
            save_fig("05c_logroi_by_quarter")

        best_month = (
            m.groupby("release_month")["log_roi"].median().sort_values(ascending=False).head(3)
        )
        print("Top months by median log_roi:")
        display(best_month.to_frame("median_log_roi"))




### Insights — release timing

- **Observation:** Month/quarter charts show concentration of releases and shifts in **median** log ROI.
- **Business interpretation:** Summer and holiday corridors may concentrate hits — or simply
  bigger films; interpretation needs genre/budget controls in modeling.
- **Modeling implication:** Use `release_month` / `release_quarter` as categorical features; watch leakage from future calendar if simulating true pre-release prediction for a fixed “today”.


## 6. Runtime

**Business lens:** Runtime signals format and production scope (epic vs tight narrative).


In [ ]:
if df is None or "runtime" not in df.columns:
    print("Skip.")
else:
    r = df.dropna(subset=["runtime"]).copy()

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(r["runtime"], bins=40, kde=True, color="slategray", ax=ax)
    ax.set_title("Runtime distribution")
    ax.set_xlabel("Runtime (minutes)")
    save_fig("06a_runtime_hist")

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=r.sample(min(2000, len(r))), x="runtime", y="log_roi", alpha=0.35, ax=ax)
    ax.set_title("Runtime vs log ROI (sample for readability)")
    ax.set_xlabel("Runtime")
    ax.set_ylabel("log_roi")
    save_fig("06b_runtime_vs_logroi")

    fig, ax = plt.subplots(figsize=(10, 5.5))
    sns.boxplot(data=r, x="movie_success_class", y="runtime", hue="movie_success_class", palette="pastel", dodge=False, legend=False, ax=ax)
    ax.set_title("Runtime by success class")
    ax.set_xlabel("movie_success_class")
    save_fig("06c_runtime_by_class")




### Insights — runtime

- **Observation:** Relationship between runtime and log ROI is usually **weak** versus budget/genre.
- **Business interpretation:** Very short/long runtimes may map to specific formats (horror vs epic) with different risk.
- **Modeling implication:** Keep `runtime` as a numeric baseline; consider non-linear transforms only if EDA shows gains.


## 7. Original language

**Business lens:** Language proxies **addressable market** vs niche/international positioning.


In [ ]:
if df is None or "original_language" not in df.columns:
    print("Skip.")
else:
    lang = df.dropna(subset=["original_language"]).copy()
    top_lang = lang["original_language"].value_counts().head(12)

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(x=top_lang.values, y=top_lang.index.astype(str), color="steelblue", ax=ax)
    ax.set_title("Most common original_language codes (top 12)")
    ax.set_xlabel("Count")
    save_fig("07a_language_counts")

    min_n = 40
    vc = lang["original_language"].value_counts()
    keep = vc[vc >= min_n].index
    lf = lang[lang["original_language"].isin(keep)]
    fig, ax = plt.subplots(figsize=(10, 5.5))
    sns.boxplot(data=lf, x="original_language", y="log_roi", hue="original_language", palette="Set2", dodge=False, legend=False, ax=ax)
    ax.set_title(f"log ROI by language (n ≥ {min_n} per language)")
    ax.set_xlabel("original_language")
    plt.xticks(rotation=0)
    save_fig("07b_logroi_by_language")

    lang["english"] = np.where(lang["original_language"] == "en", "English", "Non-English")
    fig, ax = plt.subplots(figsize=(7, 5.5))
    sns.violinplot(data=lang, x="english", y="log_roi", hue="english", palette=["#4C72B0", "#DD8452"], dodge=False, inner="box", ax=ax, legend=False)
    ax.set_title("log ROI: English vs non-English originals")
    save_fig("07c_english_vs_nonenglish_logroi")

    summ = lang.groupby("english")["log_roi"].agg(["median", "mean", "count"])
    display(summ)





### Insights — language

- **Observation:** English-heavy datasets often hide selection bias (US-centric catalog).
- **Business interpretation:** Non-English films may show different ROI mixes due to budget levels and markets served.
- **Modeling implication:** Treat `original_language` as categorical; interpret English vs rest carefully in stakeholder messaging.


## 8. Production footprint

**Business lens:** Company and country counts proxy **coalition size** and **international co-production complexity**.


In [ ]:
def parse_json_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        try:
            out = json.loads(s)
            return out if isinstance(out, list) else []
        except json.JSONDecodeError:
            return []
    return []


if df is None:
    print("Skip.")
else:
    if "production_companies" in df.columns:
        names = []
        for v in df["production_companies"]:
            for d in parse_json_list(v):
                if isinstance(d, dict) and "name" in d:
                    names.append(d["name"])
        top_co = pd.Series(names).value_counts().head(15)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(top_co.index.astype(str), top_co.values, color="#9b2226", edgecolor="white", linewidth=0.5)
        ax.invert_yaxis()
        ax.set_title("Top production companies (name frequency in dataset)")
        ax.set_xlabel("Appearances (movies can share companies)")
        save_fig("08a_top_production_companies")
        display(top_co.to_frame("appearances"))
    else:
        print("No production_companies column in CSV.")

    for col, cap, fname in [
        ("production_company_count", 12, "08b_production_company_count"),
        ("production_country_count", 8, "08c_production_country_count"),
    ]:
        if col not in df.columns:
            print("Missing", col)
            continue
        t = df[[col, "log_roi"]].dropna().copy()
        t[col] = t[col].clip(upper=cap)
        fig, ax = plt.subplots(figsize=(11, 5))
        sns.boxplot(data=t, x=col, y="log_roi", hue=col, palette="flare", dodge=False, legend=False, ax=ax)
        ax.set_title(f"log ROI vs {col} (values capped at {cap} for readability)")
        save_fig(fname)





### Insights — production structure

- **Observation:** Frequent studio banners reflect franchise machines; count features capture **co-production breadth**.
- **Business interpretation:** More countries/companies can mean risk-sharing — or coordination overhead; effect direction is empirical.
- **Modeling implication:** Use counts as numeric features; studio names need aggregation / frequency encoding later to avoid high cardinality.


## 9. Correlation heatmap (numeric drivers)

**Business lens:** Where do **pre-release numeric signals** move together with **outcomes**?
Note: `roi` / `log_roi` are outcomes — useful for EDA, **not** for leakage-safe feature matrices.


In [ ]:
if df is None:
    print("Skip.")
else:
    cols = [
        "budget",
        "runtime",
        "genre_count",
        "production_company_count",
        "production_country_count",
        "spoken_language_count",
        "roi",
        "log_roi",
    ]
    have = [c for c in cols if c in df.columns]
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("Skipping missing columns:", miss)
    cm = df[have].dropna()
    corr = cm.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True, linewidths=0.5)
    ax.set_title("Correlation matrix (numeric EDA set)")
    save_fig("09_correlation_heatmap")
    display(corr)



### Insights — correlations

- **Observation:** `budget` often correlates with `revenue` proxies indirectly via ROI structure; runtime/counts are usually weaker.
- **Business interpretation:** Large correlations among **inputs** suggest multicollinearity for linear models; low correlation with ROI
  does not mean non-linear models cannot learn patterns.
- **Modeling implication:** For **classification**, rely on validation scores; for **regression** on ROI, tree ensembles often handle skew and interactions well in a later phase.


In [ ]:
if df is None:
    print("Run notebook 01 first to populate summary statistics.")
else:
    lines = []
    lines.append("## Auto-generated summary stats (for the closing narrative)")
    vc = df["movie_success_class"].value_counts(normalize=True, dropna=True).sort_index() * 100
    lines.append("Class shares (%): " + ", ".join(f"{k}: {v:.1f}" for k, v in vc.items()))
    if "release_month" in df.columns:
        bm = df.groupby(df["release_month"].dropna().astype(int))["log_roi"].median().idxmax()
        lines.append(f"Month with highest median log_roi (coarse): {int(bm)}")
    if "main_genre" in df.columns:
        mg = df.dropna(subset=["main_genre"]).copy()
        hr = (
            mg.assign(_hit=(mg["movie_success_class"] == "hit").astype(float))
            .groupby("main_genre", observed=True)["_hit"]
            .mean()
        )
        top_hit = hr.sort_values(ascending=False).head(3)
        lines.append("Top genres by hit-rate (may be low-n): " + ", ".join(top_hit.index.astype(str)))
    print("\n".join(lines))



## Executive wrap-up (for slides / Streamlit storyline)

### Key business findings
- ROI is **heavily skewed**; success classes summarize commercial outcomes in stakeholder-friendly buckets.
- **Budget** and **genre** typically dominate exploratory separation vs timing/runtime/language — but exact ranking **depends on this run’s plots**.
- Release **month/quarter** effects are plausible and easy to explain to investors as “window risk”.

### Most promising predictive signals (hypotheses for the next phase)
- `budget`, `main_genre` / genre encodings, `release_month` or `release_quarter`, `runtime`, language, production **counts**.
- Non-linear interactions (e.g. high budget × family animation) are prime candidates for tree-based models later.

### Surprising findings to watch for
- **High-budget, low-ROI** clusters (efficiency traps) vs **low-budget breakout hits**.
- Genres with **high variance** in ROI (risky greenlights) vs stable mid-return genres.

### Future feature engineering ideas
- Multi-genre flags from `genres` JSON; frequency-encoded **studio** tiers; decade buckets from `release_year`.
- Optional: inflation-adjusted budget if you bring external indices (keep scope controlled for the course).

**Figures saved for reuse:** see `plots/business_eda/*.png` (paths printed in setup).
